But: à partir d'un contexte de risque donné par l'utilisateur, renvoie les risques correspondant lié au révisorat d'entreprise

Prérequis: disposer d'une clé OpenAI

Fonctionnement:
- Compléter le fichier de configuration config.yaml avec la bonne clé et le model choisi
- Executer la partie config
- Exécuter la partie test avec des contextes donnés. 2 possibilités soit placer le contenu du contexte dans une variable comme le premier exemple avec la variable contexte_test ou, comme le deuxième exemple, donner le contexte directement dans la variable d'appel "contexte" de la fonction

# Setup

Load packages

In [2]:
from utils import read_yaml
from pydantic import BaseModel, Field
from typing import Literal
from enum import Enum # make sure to import Enum
from openai import OpenAI
import json
import pandas as pd
import instructor

Création de la classe acceuillant les risques associés au contexte donné

In [3]:
severites_risque = ["Elevé","Moyen","Faible"]

SeveriteRisque = Enum('SeveriteRisque', {str(item): str(item) for item in severites_risque})

class Risque(BaseModel):
    #type_risque: TypeRisque = Field(..., description="type de risque identifié")
    type_risque: str = Field(..., description="intitulé risque identifié")
    risque_connu: bool = Field(..., description="indique si le type de risque est dans la liste fournie")
    label_risque: str = Field(..., description="description générale de ce type de risque sans tenir compte du contexte donné")
    raison_risque: str = Field(description="justification du choix de ce risque dans ce contexte")
    severite_risque: SeveriteRisque = Field(description="niveau de probabilité ce risque dans ce contexte")
    raison_severite: str = Field(description="justification du choix de cette probabilité dans ce contexte")

class Ensemble_Risques(BaseModel):
    Risques: list[Risque] = Field(..., description="Liste des risques associés au contexte donné") # Use list[Tag] for type hinting

Initialisation du client OpenAI

In [4]:
config = read_yaml("../config.yaml")
openai_key = config["openai_key"]

# Pass the API key to the OpenAI constructor
client = instructor.from_openai(OpenAI(api_key=openai_key))

Prompt donné au LLM

In [5]:
prompt = '''
#INSTRUCTIONS
Dans le cadre du contexte décrit ci-dessous, définis les risques liés à la mission du réviseur d'entreprise.
Pour chaque risque identifié, définis s'il sagit d'un risque repris dans les exemples ci-dessous ou un nouveau type de risque non répertorié, une description générale de ce type de risque hors contexte donné, une justification du choix de ce risque dans ce contexte, le niveau de probabilité de ce risque dans ce contexte et une justification du choix de cette probabilité dans ce contexte.
Pour le niveau de probabilité utilise la liste fournie ci-dessous.
Les exemples de risque sont les suivants:
{infos_risque}
Les niveaux de probabilité sont les suivants:
{severites_risque}

#CONTEXTE
{contexte}
#ETAPES:
1. Identifie tous les risques liés à à la mission du réviseur d'entreprise pour ce contexte.
2. Pour chaque risque identifié, définis son intitulé, s'il sagit d'un risque défini dans les exemples donnés ou un nouveau type de risque non répertorié, une description générale de ce type de risque sans tenir compte du contexte donné, une justification du choix de ce risque dans ce contexte, niveau de probabilité ce risque dans ce contexte et une justification du choix de cette probabilité dans ce contexte.
'''

Fonction d'appel du LLM tenant compte des risques précédemment identifiés

In [6]:
def question_risque(llm_name, db_risque, contexte, prompt):
  instructor_prompt = prompt.format(
      infos_risque=str(db_risque),
      severites_risque=str(severites_risque),
      contexte=contexte
      )

  response = client.chat.completions.create(
        model=llm_name,
        messages=[
            {
                "role": "user",
                "content": instructor_prompt
            }
        ],
        response_model=Ensemble_Risques)  # Specify the structured model
  
  doc_dic = json.loads(response.model_dump_json())

  # mettre à jour la DB risque
  for i in doc_dic["Risques"]:
    if i["type_risque"] not in db_risque.keys():
      db_risque[i["type_risque"]] = i["label_risque"]

  return doc_dic,db_risque

Fonction principale d'appel de question_risque et traitement du résultat

In [7]:
def call_risque(llm_name, db_risque, contexte, prompt):
  print("Liste actuelle des risques", db_risque)
  resultat = question_risque(llm_name=llm_name, db_risque=db_risque, contexte=contexte, prompt=prompt)
  db_risque = resultat[1]
  print("Liste mise à jour des risques", resultat[1])

  # Mise sous dataframe du resultat de l'analyse pour affichage
  df = pd.DataFrame(resultat[0]["Risques"])
  pd.set_option("display.max_columns", None)
  pd.set_option("display.max_colwidth", None) # Added this line
  return db_risque, df

# Tests

In [8]:
contexte_test = """Quels sont les risques liés au contexte ci-dessous: "Un réviseur d’entreprise doit faire un audit et un rapport dans le cadre d’une liquidation.
La société est une SA en Belgique et son activité est : holding
Le type d’actionnariat est fermé
Le dirigeant est unique
Il n’y a pas d'Indication de comportement non éthique des dirigeants
La Nature des opérations de l’entité est simple : mono produit
Il y a un schéma classique des cycles (Achat, vente, salaire, financiers)
La méthode comptable est : Belgian gaap
Il n’y a pas de complexité comptable
L’expert-comptable est certifié ITAA et le réviseur à l'habitude de travailler"""

In [ ]:
db_risque = {}
db_risque, df = call_risque(llm_name=config["llm_chosen"], db_risque=db_risque, contexte=contexte_test, prompt=prompt)
df.style.set_properties(**{'text-align': 'left'})

In [ ]:
types_risque, df = call_risque(llm_name=config["llm_chosen"], db_risque=db_risque, contexte="boulangerie", prompt=prompt)
df.style.set_properties(**{'text-align': 'left'})

Pour tout nouveau test, changer `context_var` soit par le nom d'une variable textuelle contenant le contexte soit placer directement le contexte comme variable texte

In [ ]:
context_var = ...

In [ ]:
types_risque,df=call_risque(llm_name=config["llm_chosen"], db_risque=db_risque, contexte=context_var, prompt=prompt)
df.style.set_properties(**{'text-align': 'left'})